In [3]:
# Using PyTorch 1.4

import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import random
import torch
import fire
import logging
import os
import csv

from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from torch.optim import SGD
from tqdm import tqdm, trange
import torch.nn.functional as F

E0000 00:00:1740246532.778794     295 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740246532.783696     295 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [12]:
class CSVMIMIC(Dataset):
    
    def __init__(self, control_code, truncate=False, gpt2_type="gpt2", max_length=768):

        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        self.notes = []

        # This uses the MIMIC3 csv
        
        with open('MIMIC_NoteEvents_deanonymized.csv', newline='', encoding='ISO-8859-1') as csvfile:
            tweet_csv = csv.reader(csvfile)
            for row in tweet_csv:
                self.notes.append(torch.tensor(
                    self.tokenizer.encode(f"<|{control_code}|>{row[10][:max_length]}<|endoftext|>")
                ))
                
        if truncate:
            self.notes = self.notes[:20000]
        self.notes_count = len(self.notes)
        
    def __len__(self):
        return self.notes_count

    def __getitem__(self, item):
        return self.notes[item]



In [5]:
def pack_tensor(new_tensor, packed_tensor, max_seq_len):
    if packed_tensor is None:
        return new_tensor, True, None
    if new_tensor.size()[1] + packed_tensor.size()[1] > max_seq_len:
        return packed_tensor, False, new_tensor
    else:
        packed_tensor = torch.cat([new_tensor, packed_tensor[:, 1:]], dim=1)
        return packed_tensor, True, None

In [16]:
def train(
    dataset,
    model,
    tokenizer,
    batch_size=16,
    epochs=4,
    lr=2e-5,
    max_seq_len=400,
    warmup_steps=5000,
    gpt2_type="gpt2",
    device="cuda",
    output_dir=".",
    output_prefix="wreckgar",
    test_mode=False,
    save_model_on_epoch=False,
):

    acc_steps = 100

    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=-1
    )

    train_dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    accumulating_batch_count = 0
    input_tensor = None

    for epoch in range(epochs):

        print(f"Training epoch {epoch}")
        for idx, entry in tqdm(enumerate(train_dataloader)):
            (input_tensor, carry_on, remainder) = pack_tensor(entry, input_tensor, 768)

            if carry_on and idx != len(train_dataloader) - 1:
                continue

            input_tensor = input_tensor.to(device)
            outputs = model(input_tensor, labels=input_tensor)
            loss = outputs[0]
            loss.backward()

            if (accumulating_batch_count % batch_size) == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                model.zero_grad()

            accumulating_batch_count += 1
            input_tensor = None
        if save_model_on_epoch:
            torch.save(
                model.state_dict(),
                os.path.join(output_dir, f"{output_prefix}-{epoch}.pt"),
            )
    return model

In [13]:
dataset = CSVMIMIC("<|note|>", truncate=True, gpt2_type="gpt2")

In [8]:
gpt2_type = "gpt2"

In [9]:
YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [ ]:
learning_rates = [3e-9, 3e-8, 3e-6, 3e-5, 3e-4, 3e-2]
batch_sizes= [8,16,32,64]
warmup_steps= [1000, 2500, 5000, 8000]
for i in range(len(learning_rates)):
    model = train(
        dataset,
        GPT2LMHeadModel.from_pretrained(gpt2_type),
        GPT2Tokenizer.from_pretrained(gpt2_type),
        batch_size=16,
        epochs=1,
        lr=learning_rates[i],
        max_seq_len=140,
        warmup_steps=5000,
        gpt2_type=gpt2_type,
        device="cuda",
        output_dir="trained_models",
        output_prefix="note",
        save_model_on_epoch=False
    )
    modelDir = 'model'+str(i)
    model.save_pretrained(modelDir)
    tokenizer.save_pretrained(modelDir)
    result = !lm_eval \
    --model hf \
    --model_args pretrained=modelDir \
    --output_path /Output \
    --include_path ./ \
    --tasks multimedqa \
    --device cpu \
    --trust_remote_code \
    fileName = str( "trial" + str(i) + ".yaml")
    with open(fileName, "w") as f:
        f.write(str(result[76:]))


Training epoch 0


12it [00:00, 31.60it/s]

In [ ]:
model.save_pretrained('model')

In [ ]:
tokenizer=GPT2Tokenizer.from_pretrained(gpt2_type)
tokenizer.save_pretrained('model')

In [ ]:
#Results without MIMIC3

"""|             Tasks             |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|-------------------------------|-------|------|-----:|--------|---|----:|---|-----:|
|multimedqa                     |    N/A|none  |      |acc     |↑  |  0.3|±  |0.0490|
| - medmcqa                     |Yaml   |none  |     0|acc     |↑  |  0.5|±  |0.1667|
|                               |       |none  |     0|acc_norm|↑  |  0.5|±  |0.1667|
| - medqa_4options              |Yaml   |none  |     0|acc     |↑  |  0.1|±  |0.1000|
|                               |       |none  |     0|acc_norm|↑  |  0.1|±  |0.1000|
| - anatomy (mmlu)              |      1|none  |     0|acc     |↑  |  0.3|±  |0.1528|
| - clinical_knowledge (mmlu)   |      1|none  |     0|acc     |↑  |  0.4|±  |0.1633|
| - college_biology (mmlu)      |      1|none  |     0|acc     |↑  |  0.3|±  |0.1528|
| - college_medicine (mmlu)     |      1|none  |     0|acc     |↑  |  0.4|±  |0.1633|
| - medical_genetics (mmlu)     |      1|none  |     0|acc     |↑  |  0.3|±  |0.1528|
| - professional_medicine (mmlu)|      1|none  |     0|acc     |↑  |  0.1|±  |0.1000|
| - pubmedqa                    |      1|none  |     0|acc     |↑  |  0.3|±  |0.1528|

|  Groups  |Version|Filter|n-shot|Metric|   |Value|   |Stderr|
|----------|-------|------|------|------|---|----:|---|-----:|
|multimedqa|    N/A|none  |      |acc   |↑  |  0.3|±  | 0.049|"""